In [ ]:
import json

with open("<TODO>/basque_linguistic_competence/logs/data-all.json", "r", encoding="utf-8") as f:
    data = json.loads(f.read())

del f

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# --- 1. Data Setup ---
bench, bat, model = [], [], []
for key, value in data.items():
    if "_60M-2-new" in key:
        continue
    model.append(key)
    if "bat-ctxt" in value["LOGPROBS"]:
        bat.append(np.mean(value["LOGPROBS"]["bat-ctxt"]) * 100)
    else:
        bat.append(np.mean(value["LOGPROBS"]["bat-inst"]) * 100)

    bench.append(
        (
            value["BENCHMARK"]["arc_eu_challenge_mc"]
        +
            value["BENCHMARK"]["arc_eu_easy_mc"]
        +
            value["BENCHMARK"]["belebele_eus_Latn"]
        +
            value["BENCHMARK"]["bertaqa_eu_global"]
        +
            value["BENCHMARK"]["bertaqa_eu_local"]
        +
            value["BENCHMARK"]["eus_exams_eu"]
        +
            value["BENCHMARK"]["eus_reading"]
        +
            value["BENCHMARK"]["eus_trivia"]
        +
            value["BENCHMARK"]["mmlu_eu"]
        +
            value["BENCHMARK"]["piqa_eu_mc"]
        )
        /
        10
        *
        100
    )

del key, value

# Boolean mask to determine which points get labels (True = label, False = no label)
mask = [False] * len(bat)

assert len(bat) == len(bench) == len(model) == len(mask), f"{len(bat)}, {len(bench)}, {len(model)}, {len(mask)}"


# --- 2. Formal Styling ---
sns.set_style("whitegrid")
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 14,           # Increased general font size
    'axes.labelsize': 16,      # Decreased from 20 to make axis letters smaller
    'axes.titlesize': 18,
    'xtick.labelsize': 14,     # Increased tick size
    'ytick.labelsize': 14,     # Increased tick size
    'figure.dpi': 300,
    'grid.linestyle': '--',
    'grid.alpha': 0.6
})

fig, ax = plt.subplots(figsize=(6, 6))

# --- 3. Updated Function ---
def plot_group(x, y, color, point_labels=None, label_mask=None):
    # Changed alpha to 1.0, edgecolor to black, increased size (s=100) for stronger dots
    ax.scatter(x, y, c=color, s=100, alpha=1.0, 
               edgecolor='black', linewidth=1.5, zorder=3)
    
    if point_labels and label_mask:
        for px, py, txt, show_label in zip(x, y, point_labels, label_mask):
            if show_label:
                ax.annotate(
                    txt, 
                    (px, py), 
                    xytext=(5, 5), 
                    textcoords='offset points', 
                    fontsize=9, 
                    color='black',
                    alpha=0.8
                )

# --- 4. Plotting ---
# Using a strong Midnight Blue for extreme contrast and professional look
plot_group(bat, bench, color='#191970', point_labels=model, label_mask=mask)

# --- 5. Final Formatting ---
ax.set_xlim([50, 80])
ax.set_ylim([50, 85])

# Adding bold weights to make the large letters stand out even more
ax.set_ylabel("General-Purpose Benchmark Average Score", weight='bold')
ax.set_xlabel("BAT Accuracy Score", weight='bold')

sns.despine(trim=True, offset=5)

# Removed the legend block entirely

plt.tight_layout(pad=0)
plt.savefig("BAT-benchmarks.pdf", format="pdf", bbox_inches="tight", pad_inches=0)
plt.show()

# Removed item and leg from deletion since they no longer exist
del bat, bench, model, mask, fig, ax

In [ ]:
import numpy as np
from scipy import stats

pairs = [
    ["swiss-ai/Apertus-8B-2509", "swiss-ai/Apertus-70B-2509"],
    ["google/gemma-3-12b-pt", "google/gemma-3-27b-pt", "google/gemma-4-12B", "google/gemma-4-26B-A4B", "google/gemma-4-31B"],
    ["zai-org/GLM-4-32B-Base-0414", "zai-org/GLM-4.5-Air-Base"],
    ["meta-llama/Llama-3.1-8B", "meta-llama/Llama-3.1-70B", "meta-llama/Llama-4-Scout-17B-16E"],
    ["Qwen/Qwen3.5-9B-Base", "Qwen/Qwen3.5-35B-A3B-Base"],
    ["BSC-LT/salamandra-7b", "BSC-LT/ALIA-40b"],
    ["orai-nlp/Gemma-Kimu-9b-base", "orai-nlp/Llama-eus-8B"],
    ["HiTZ/Latxa-Llama-3.1-8B"],
]

sent, inst = [], []

for pair in pairs:

    cont = []
    for aux in pair:
        cont.append(data[aux]["LOGPROBS"]["bat-sent"])
    cont = np.mean(cont, axis=0).tolist()
    sent.append(cont)

    cont = []
    for aux in pair:
        cont.append(data[aux]["LOGPROBS"]["bat-ctxt"])
    cont = np.mean(cont, axis=0).tolist()
    inst.append(cont)

for pair, s, i in zip(pairs, sent, inst):
    mean_diff = np.mean(s) - np.mean(i)
    _, p_value = stats.ttest_rel(s, i)
    
    if p_value > 0.05:
        angle = "\\sim"
        # print("Not Significant (p > 0.05)")      
    elif 0.01 < p_value <= 0.05:
        angle = "\\subtleup" if - mean_diff > 0 else "\\subtledown"
        # print("Significant Increase" if mean_diff > 0 else "Significant Drop")
        
    else:
        angle = "\\nearrow" if - mean_diff > 0 else "\\searrow"
        # print("Highly Sig. Increase" if mean_diff > 0 else "Highly Sig. Drop")

    print(pair, f"& {np.mean(s) * 100:.2f} \\({angle}\\) {np.mean(i) * 100:.2f} \\,\\textcolor{{{"darkred" if mean_diff > 0.0 else "darkgreen"}}}{{\\scriptsize ({- mean_diff * 100:+.2f})}} \\\\")

del pairs, sent, inst, pair, s, i, aux, cont, p_value, mean_diff, angle

In [ ]:
import numpy as np
from scipy import stats

pairs = [
    [
        ["swiss-ai/Apertus-8B-2509", "swiss-ai/Apertus-70B-2509"],
        ["swiss-ai/Apertus-8B-Instruct-2509", "swiss-ai/Apertus-70B-Instruct-2509"]
    ],
    [
        ["google/gemma-3-12b-pt", "google/gemma-3-27b-pt", "google/gemma-4-12B", "google/gemma-4-26B-A4B", "google/gemma-4-31B"],
        ["google/gemma-3-12b-it", "google/gemma-3-27b-it", "google/gemma-4-12B-it", "google/gemma-4-26B-A4B-it", "google/gemma-4-31B-it"]
    ],
    [
        ["zai-org/GLM-4-32B-Base-0414", "zai-org/GLM-4.5-Air-Base"],
        ["zai-org/GLM-4-32B-0414", "zai-org/GLM-4.5-Air"]
    ],
    [
        ["meta-llama/Llama-3.1-8B", "meta-llama/Llama-3.1-70B", "meta-llama/Llama-4-Scout-17B-16E"],
        ["meta-llama/Llama-3.1-8B-Instruct", "meta-llama/Llama-3.1-70B-Instruct", "meta-llama/Llama-4-Scout-17B-16E-Instruct"]
    ],
    [
        ["Qwen/Qwen3.5-9B-Base", "Qwen/Qwen3.5-35B-A3B-Base"],
        ["Qwen/Qwen3.5-9B", "Qwen/Qwen3.5-35B-A3B"]
    ],
    [
        ["BSC-LT/salamandra-7b", "BSC-LT/ALIA-40b"],
        ["BSC-LT/salamandra-7b-instruct", "BSC-LT/ALIA-40b-instruct-2605"]
    ],
    [
        ["orai-nlp/Gemma-Kimu-9b-base", "orai-nlp/Llama-eus-8B"],
        ["orai-nlp/Gemma-Kimu-9b-it", "orai-nlp/Llama-eus-8B-DIPLomA"]
    ],
    [
        ["HiTZ/Latxa-Llama-3.1-8B"],
        ["HiTZ/Latxa-Llama-3.1-8B-Instruct"]
    ],
]

base, instruct = [], []


for pair in pairs:
    b, i = pair[0], pair[1]

    cont = []
    for aux in b:
        cont.append(data[aux]["LOGPROBS"]["bat-ctxt"])
    cont = np.mean(cont, axis=0).tolist()
    base.append(cont)

    cont = []
    for aux in i:
        cont.append(data[aux]["LOGPROBS"]["bat-inst"])
    cont = np.mean(cont, axis=0).tolist()
    instruct.append(cont)

for pair, b, i in zip(pairs, base, instruct):
    mean_diff = np.mean(b) - np.mean(i)
    _, p_value = stats.ttest_rel(b, i)
    
    if p_value > 0.05:
        angle = "\\sim"
        # print("Not Significant (p > 0.05)")      
    elif 0.01 < p_value <= 0.05:
        angle = "\\subtleup" if - mean_diff > 0 else "\\subtledown"
        # print("Significant Increase" if mean_diff > 0 else "Significant Drop")
        
    else:
        angle = "\\nearrow" if - mean_diff > 0 else "\\searrow"
        # print("Highly Sig. Increase" if mean_diff > 0 else "Highly Sig. Drop")

    print(pair, f"& {np.mean(b) * 100:.2f} \\({angle}\\) {np.mean(i) * 100:.2f} \\,\\textcolor{{{"darkred" if mean_diff > 0.0 else "darkgreen"}}}{{\\footnotesize ({- mean_diff * 100:+.2f})}} \\\\")

del pairs, base, instruct, pair, b, i, aux, cont, p_value, mean_diff, angle

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np


# --- 1. Data Preparation ---
gens, discs = [], []
for key, value in data.items():
    if "_60M-2-new" in key:
        continue
    
    if "bat-ctxt" in value["LOGPROBS"]:
        continue
        # gens.append(np.mean(value["LOGPROBS"]["bat-ctxt"]) * 100)
        # discs.append(np.mean(value["BENCHMARK"]["ab-base_BAT"]) * 100)
    else:
        gens.append(np.mean(value["LOGPROBS"]["bat-inst"]) * 100)
        discs.append(np.mean(value["BENCHMARK"]["ab-instruct_BAT"]) * 100)
    
    if discs[-1] > gens[-1]:
        print(key, gens[-1], discs[-1])


assert len(gens) == len(discs)


# --- 2. Formal Styling ---
sns.set_style("whitegrid")
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 12,
    'axes.labelsize': 20, # CHANGED: Increased further to 20
    'axes.titlesize': 14,
    'figure.dpi': 300,
    'grid.linestyle': '--',
    'grid.alpha': 0.6
})


fig, ax = plt.subplots(figsize=(6, 6))


# --- NEW: Fill Backgrounds ---
x_fill = np.array([40, 100])


# Top-Left Triangle (y > x) - Stronger Yellow/Gold Background
ax.fill_between(x_fill, x_fill, 100, color='#FFE082', alpha=0.55, zorder=0)


# Bottom-Right Triangle (y < x) - Stronger Blue Background
ax.fill_between(x_fill, 40, x_fill, color='#B3E5FC', alpha=0.55, zorder=0)


# --- 3. Plotting Scatters ---
# Changed to Dark Slate for optimal contrast against both backgrounds and the red trendline
ax.scatter(discs, gens, c='black', s=80, alpha=0.9, 
           edgecolor='white', linewidth=1.2, zorder=3)


# --- 4. Aesthetic Trendline over the dots ---
m, b = np.polyfit(discs, gens, 1) 


# Keep the moderate physical length (extends just slightly past the dots)
x_vals = np.array([min(discs) - 2, max(discs) + 2]) 
y_vals = m * x_vals + b


# Add a subtle white shadow/glow behind the line for better readability
ax.plot(x_vals, y_vals, color='white', linestyle='-', linewidth=4, zorder=4, alpha=0.7)


# The main aesthetic trendline
line, = ax.plot(x_vals, y_vals, color='#D32F2F', linestyle=(0, (2, 2)), linewidth=2.5, zorder=5, label="BAT Interpolation")


# --- 5. Diagonal Arrow for Stronger Model ---
ax.annotate('', xy=(94, 94), xycoords='data', 
            xytext=(45, 45), textcoords='data',
            arrowprops=dict(arrowstyle="->", color='#888888', linewidth=1.5, linestyle=':', shrinkA=0, shrinkB=0),
            zorder=1)


ax.text(86, 87.5, "Strongest Model", rotation=45, ha='center', va='center', 
        fontsize=15, color='black', fontweight='bold', alpha=0.8, zorder=2)


# Aesthetic Region Labels
ax.text(58, 85, "GEN better than DISC", rotation=45, ha='center', va='center', 
        fontsize=15, color='black', fontweight='bold', alpha=0.8, zorder=2)
ax.text(82, 58, "DISC better than GEN", rotation=45, ha='center', va='center', 
        fontsize=15, color='black', fontweight='bold', alpha=0.8, zorder=2)


# --- NEW: Small Legend specifically for the trendline ---
ax.legend(handles=[line], loc='lower right', fontsize=10, 
          frameon=True, facecolor='white', edgecolor='#CCCCCC', 
          framealpha=0.9, borderpad=0.8, shadow=True)


# --- 6. Final Formatting ---
ax.set_xlim([45, 95])
ax.set_ylim([45, 95])

# CHANGED: Font size increased to 20 and weight set to bold
ax.set_xlabel("A/B Multiple-Choice", fontsize=17, fontweight='bold')
ax.set_ylabel("Instruction Probability Measurements", fontsize=17, fontweight='bold')
ax.set_aspect('equal')


# Despine and trim based on instructions
sns.despine(trim=True, offset=5)


plt.tight_layout(pad=0)
plt.savefig("BAT.pdf", format="pdf", bbox_inches="tight", pad_inches=0)
plt.show()


del ax, b, discs, fig, gens, key, line, m, value, x_fill, x_vals, y_vals

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np


# --- 1. Data Preparation ---
gens, discs = [], []
for key, value in data.items():
    if "_60M-2-new" in key:
        continue
    
    if "bat-ctxt" in value["LOGPROBS"]:
        continue
        # gens.append(np.mean(value["LOGPROBS"]["bl2mp-ctxt"]) * 100)
        # discs.append(np.mean(value["BENCHMARK"]["ab-base_BL2MP"]) * 100)
    else:
        gens.append(np.mean(value["LOGPROBS"]["bl2mp-inst"]) * 100)
        discs.append(np.mean(value["BENCHMARK"]["ab-instruct_BL2MP"]) * 100)
    
    if discs[-1] > gens[-1]:
        print(key, gens[-1], discs[-1])
    if discs[-1] > 80.0:
        print(key, gens[-1], discs[-1])
        print()



assert len(gens) == len(discs)


# --- 2. Formal Styling ---
sns.set_style("whitegrid")
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 12,
    'axes.labelsize': 20, # CHANGED: Increased further to 20
    'axes.titlesize': 14,
    'figure.dpi': 300,
    'grid.linestyle': '--',
    'grid.alpha': 0.6
})


fig, ax = plt.subplots(figsize=(6, 6))


# --- NEW: Fill Backgrounds ---
x_fill = np.array([40, 100])


# Top-Left Triangle (y > x) - Stronger Yellow/Gold Background
ax.fill_between(x_fill, x_fill, 100, color='#FFE082', alpha=0.55, zorder=0)


# Bottom-Right Triangle (y < x) - Stronger Blue Background
ax.fill_between(x_fill, 40, x_fill, color='#B3E5FC', alpha=0.55, zorder=0)


# --- 3. Plotting Scatters ---
# Changed to black for optimal contrast against both backgrounds and the red trendline
ax.scatter(discs, gens, c='black', s=80, alpha=0.9, 
           edgecolor='white', linewidth=1.2, zorder=3)


# --- 4. Aesthetic Trendline over the dots ---
m, b = np.polyfit(discs, gens, 1) 


# Keep the moderate physical length (extends just slightly past the dots)
x_vals = np.array([min(discs) - 2, max(discs) + 2]) 
y_vals = m * x_vals + b


# Add a subtle white shadow/glow behind the line for better readability
ax.plot(x_vals, y_vals, color='white', linestyle='-', linewidth=4, zorder=4, alpha=0.7)


# The main aesthetic trendline
line, = ax.plot(x_vals, y_vals, color='#D32F2F', linestyle=(0, (2, 2)), linewidth=2.5, zorder=5, label="BL2MP Interpolation")


# --- 5. Diagonal Arrow for Stronger Model ---
ax.annotate('', xy=(94, 94), xycoords='data', 
            xytext=(45, 45), textcoords='data',
            arrowprops=dict(arrowstyle="->", color='#888888', linewidth=1.5, linestyle=':', shrinkA=0, shrinkB=0),
            zorder=1)


ax.text(86, 87.5, "Strongest Model", rotation=45, ha='center', va='center', 
        fontsize=15, color='black', fontweight='bold', alpha=0.8, zorder=2)


# Aesthetic Region Labels
ax.text(58, 85, "GEN better than DISC", rotation=45, ha='center', va='center', 
        fontsize=15, color='black', fontweight='bold', alpha=0.8, zorder=2)
ax.text(82, 58, "DISC better than GEN", rotation=45, ha='center', va='center', 
        fontsize=15, color='black', fontweight='bold', alpha=0.8, zorder=2)


# --- NEW: Small Legend specifically for the trendline ---
ax.legend(handles=[line], loc='lower right', fontsize=10, 
          frameon=True, facecolor='white', edgecolor='#CCCCCC', 
          framealpha=0.9, borderpad=0.8, shadow=True)


# --- 6. Final Formatting ---
ax.set_xlim([45, 95])
ax.set_ylim([45, 95])


# CHANGED: Font size increased to 17 and weight set to bold
ax.set_xlabel("A/B Multiple-Choice", fontsize=17, fontweight='bold')
ax.set_ylabel("Instruction Probability Measurements", fontsize=17, fontweight='bold')
ax.set_aspect('equal')


# Despine and trim based on instructions
sns.despine(trim=True, offset=5)


plt.tight_layout(pad=0)
plt.savefig("BL2MP.pdf", format="pdf", bbox_inches="tight", pad_inches=0)
plt.show()


del ax, b, discs, fig, gens, key, line, m, value, x_fill, x_vals, y_vals

In [ ]:
import numpy as np
from scipy import stats

pairs = [
    ["Qwen/Qwen3.5-9B"],
    ["Qwen/Qwen3.5-27B"],
    ["Qwen/Qwen3.5-35B-A3B"],
    ["Qwen/Qwen3.5-122B-A10B"],
    ["Qwen/Qwen3.6-27B"],
    ["Qwen/Qwen3.6-35B-A3B"],
    ["google/gemma-4-12B-it"],
    ["google/gemma-4-26B-A4B-it"],
    ["google/gemma-4-31B-it"],
    ["zai-org/GLM-4.5-Air"],
    ["zai-org/GLM-4.6V"],
    ["zai-org/GLM-4.6V-Flash"],
    ["gpt-5-2025-08-07"],
    ["gpt-5.1-2025-11-13"],
]

data.setdefault("gpt-5-2025-08-07", {}).setdefault("THINKING", {})["ab-instruct_BAT-THINKING-SAMPLING"] = [0.8401585204755614, 0.8520475561426685, 0.8401585204755614, 0.8560105680317041, 0.8533685601056803]
data.setdefault("gpt-5-2025-08-07", {}).setdefault("BENCHMARK", {})["ab-instruct_BAT"] = [0.7476882430647291, 0.7397622192866579, 0.7595772787318362, 0.7384412153236459, 0.7225891677675033]

data.setdefault("gpt-5.1-2025-11-13", {}).setdefault("THINKING", {})["ab-instruct_BAT-THINKING-SAMPLING"] = [0.8414795244385733, 0.8335535006605019, 0.8467635402906208, 0.8243064729194187, 0.8269484808454426]
data.setdefault("gpt-5.1-2025-11-13", {}).setdefault("BENCHMARK", {})["ab-instruct_BAT"] = [0.7476882430647291, 0.7437252311756936, 0.7437252311756936, 0.7529722589167768, 0.7212681638044914]

inst, think = [], []

for pair in pairs:

    cont = []
    for aux in pair:
        cont.append(data[aux]["BENCHMARK"]["ab-instruct_BAT"])
    cont = np.mean(cont, axis=0).tolist()
    inst.append(cont)

    cont = []
    for aux in pair:
        # cont.append(data[aux]["THINKING"]["ab-instruct_BAT-THINKING"])
        cont.append(data[aux]["THINKING"]["ab-instruct_BAT-THINKING-SAMPLING"])
    cont = np.mean(cont, axis=0).tolist()
    think.append(cont)

for pair, i, t in zip(pairs, inst, think):
    mean_diff = np.mean(i) - np.mean(t)
    _, p_value = stats.ttest_rel(i, t)
    
    if p_value > 0.05:
        angle = "\\sim"
        # print("Not Significant (p > 0.05)")      
    elif 0.01 < p_value <= 0.05:
        angle = "\\subtleup" if - mean_diff > 0 else "\\subtledown"
        # print("Significant Increase" if mean_diff > 0 else "Significant Drop")
        
    else:
        angle = "\\nearrow" if - mean_diff > 0 else "\\searrow"
        # print("Highly Sig. Increase" if mean_diff > 0 else "Highly Sig. Drop")

    print(pair, f"& {np.mean(i) * 100:.2f} \\({angle}\\) {np.mean(t) * 100:.2f} \\,\\textcolor{{{"darkred" if mean_diff > 0.0 else "darkgreen"}}}{{\\scriptsize ({- mean_diff * 100:+.2f})}} \\\\")

del pairs, inst, think, pair, t, i, aux, cont, p_value, mean_diff, angle, data["gpt-5-2025-08-07"], data["gpt-5.1-2025-11-13"]

In [ ]:
data.setdefault("gpt-4o-2024-08-06", {}).setdefault("BENCHMARK", {})["ab-instruct_BAT"] = [0.7688243064729194, 0.7675033025099075, 0.7542932628797886, 0.7701453104359313, 0.7529722589167768]
data.setdefault("gpt-4o-2024-08-06", {}).setdefault("BENCHMARK", {})["ab-instruct_BL2MP"] = [0.8233983286908078, 0.833983286908078, 0.8144846796657381, 0.8222841225626741, 0.8272980501392758]

data.setdefault("gpt-4.1-2025-04-14", {}).setdefault("BENCHMARK", {})["ab-instruct_BAT"] = [0.7675033025099075, 0.7569352708058125, 0.7556142668428005, 0.7463672391017173, 0.7437252311756936]
data.setdefault("gpt-4.1-2025-04-14", {}).setdefault("BENCHMARK", {})["ab-instruct_BL2MP"] = [0.8245125348189415, 0.8395543175487465, 0.8323119777158774, 0.8367688022284122, 0.8222841225626741]

data.setdefault("gpt-5-2025-08-07_minimal", {}).setdefault("BENCHMARK", {})["ab-instruct_BAT"] = [0.7476882430647291, 0.7397622192866579, 0.7595772787318362, 0.7384412153236459, 0.7225891677675033]
data.setdefault("gpt-5-2025-08-07_minimal", {}).setdefault("BENCHMARK", {})["ab-instruct_BL2MP"] = [0.8272980501392758, 0.8289693593314763, 0.8250696378830084, 0.8373259052924791, 0.8328690807799443]
data.setdefault("gpt-5-2025-08-07_medium", {}).setdefault("BENCHMARK", {})["ab-instruct_BAT"] = [0.8256274768824307, 0.8401585204755614, 0.8282694848084544, 0.8467635402906208, 0.8335535006605019]
data.setdefault("gpt-5-2025-08-07_medium", {}).setdefault("BENCHMARK", {})["ab-instruct_BL2MP"] = [0.9259052924791087, 0.9303621169916435, 0.9275766016713092, 0.9292479108635098, 0.9286908077994429]
data.setdefault("gpt-5-2025-08-07_high", {}).setdefault("BENCHMARK", {})["ab-instruct_BAT"] = [0.8401585204755614, 0.8520475561426685, 0.8401585204755614, 0.8560105680317041, 0.8533685601056803]
data.setdefault("gpt-5-2025-08-07_high", {}).setdefault("BENCHMARK", {})["ab-instruct_BL2MP"] = [0.9364902506963788, 0.9259052924791087, 0.935933147632312, 0.9298050139275766, 0.9309192200557103]


data.setdefault("gpt-5.1-2025-11-13_none", {}).setdefault("BENCHMARK", {})["ab-instruct_BAT"] = [0.7476882430647291, 0.7437252311756936, 0.7437252311756936, 0.7529722589167768, 0.7212681638044914]
data.setdefault("gpt-5.1-2025-11-13_none", {}).setdefault("BENCHMARK", {})["ab-instruct_BL2MP"] = [0.8362116991643455, 0.833983286908078, 0.8284122562674094, 0.8373259052924791, 0.8362116991643455]
data.setdefault("gpt-5.1-2025-11-13_low", {}).setdefault("BENCHMARK", {})["ab-instruct_BAT"] = [0.809775429326288, 0.7912813738441216, 0.774108322324967, 0.8190224570673712, 0.8058124174372523]
data.setdefault("gpt-5.1-2025-11-13_low", {}).setdefault("BENCHMARK", {})["ab-instruct_BL2MP"] = [0.8930362116991644, 0.9013927576601671, 0.8969359331476323, 0.9002785515320334, 0.8952646239554317]
data.setdefault("gpt-5.1-2025-11-13_medium", {}).setdefault("BENCHMARK", {})["ab-instruct_BAT"] = [0.8203434610303831, 0.7833553500660502, 0.8058124174372523, 0.8110964332892999, 0.8150594451783355]
data.setdefault("gpt-5.1-2025-11-13_medium", {}).setdefault("BENCHMARK", {})["ab-instruct_BL2MP"] = [0.9075208913649025, 0.9103064066852368, 0.9164345403899722, 0.9153203342618385, 0.9075208913649025]
data.setdefault("gpt-5.1-2025-11-13_high", {}).setdefault("BENCHMARK", {})["ab-instruct_BAT"] = [0.8414795244385733, 0.8335535006605019, 0.8467635402906208, 0.8243064729194187, 0.8269484808454426]
data.setdefault("gpt-5.1-2025-11-13_high", {}).setdefault("BENCHMARK", {})["ab-instruct_BL2MP"] = [0.9298050139275766, 0.9275766016713092, 0.9264623955431754, 0.9253481894150418, 0.9275766016713092]


In [ ]:
pairs = [
    [
        ["swiss-ai/Apertus-8B-2509", "swiss-ai/Apertus-70B-2509"],
        ["swiss-ai/Apertus-8B-Instruct-2509", "swiss-ai/Apertus-70B-Instruct-2509"]
    ],
    [
        ["google/gemma-3-12b-pt", "google/gemma-3-27b-pt", "google/gemma-4-12B", "google/gemma-4-26B-A4B", "google/gemma-4-31B"],
        ["google/gemma-3-12b-it", "google/gemma-3-27b-it", "google/gemma-4-12B-it", "google/gemma-4-26B-A4B-it", "google/gemma-4-31B-it"]
    ],
    [
        ["zai-org/GLM-4-32B-Base-0414", "zai-org/GLM-4.5-Air-Base", "", "", ""],
        ["zai-org/GLM-4-32B-0414", "zai-org/GLM-4.5-Air", "zai-org/GLM-4.6V", "zai-org/GLM-4.6V-Flash", "zai-org/GLM-4.7-Flash"]
    ],
    [
        ["meta-llama/Llama-3.1-8B", "meta-llama/Llama-3.1-70B", "", "meta-llama/Llama-4-Scout-17B-16E"],
        ["meta-llama/Llama-3.1-8B-Instruct", "meta-llama/Llama-3.1-70B-Instruct", "meta-llama/Llama-3.3-70B-Instruct", "meta-llama/Llama-4-Scout-17B-16E-Instruct"]
    ],
    [
        ["Qwen/Qwen3.5-9B-Base", "Qwen/Qwen3.5-35B-A3B-Base", "", "", "", ""],
        ["Qwen/Qwen3.5-9B", "Qwen/Qwen3.5-35B-A3B", "Qwen/Qwen3.5-27B", "Qwen/Qwen3.5-122B-A10B", "Qwen/Qwen3.6-27B", "Qwen/Qwen3.6-35B-A3B"]
    ],
    [
        ["BSC-LT/salamandra-7b", "BSC-LT/ALIA-40b"],
        ["BSC-LT/salamandra-7b-instruct", "BSC-LT/ALIA-40b-instruct-2605"]
    ],
    [
        ["orai-nlp/Gemma-Kimu-9b-base", "orai-nlp/Llama-eus-8B"],
        ["orai-nlp/Gemma-Kimu-9b-it", "orai-nlp/Llama-eus-8B-DIPLomA"]
    ],
    [
        ["HiTZ/Latxa-Llama-3.1-8B", "", "", "", ""],
        ["HiTZ/Latxa-Llama-3.1-8B-Instruct", "HiTZ/Latxa-Llama-3.1-70B-Instruct", "HiTZ/Latxa-Llama-3.1-70B-Instruct-exp_2_101_v2", "HiTZ/Latxa-Qwen3-VL-8B-Instruct", "HiTZ/Latxa-Qwen3-VL-32B-Instruct"]
    ],
    [
        ["", "", "", "", "", "", "", "", ""],
        ["gpt-4o-2024-08-06", "gpt-4.1-2025-04-14", "gpt-5-2025-08-07_minimal", "gpt-5-2025-08-07_medium", "gpt-5-2025-08-07_high", "gpt-5.1-2025-11-13_none", "gpt-5.1-2025-11-13_low", "gpt-5.1-2025-11-13_medium", "gpt-5.1-2025-11-13_high"]
    ]
]

for pair in pairs:
    for base, instruct in zip(*pair):
        if base != "":
            val1 = np.mean(data[base]["LOGPROBS"]["bl2mp-ctxt"])
            val2 = np.mean(data[base]["LOGPROBS"]["bl2mp-sent"])
            val3 = np.mean(data[base]["BENCHMARK"]["ab-base_BL2MP"])
            val4 = np.mean(data[base]["LOGPROBS"]["bat-ctxt"])
            val5 = np.mean(data[base]["LOGPROBS"]["bat-sent"])
            val6 = np.mean(data[base]["BENCHMARK"]["ab-base_BAT"])
            val7 = (data[base]["BENCHMARK"]["arc_eu_challenge_mc"] + data[base]["BENCHMARK"]["arc_eu_easy_mc"] + data[base]["BENCHMARK"]["belebele_eus_Latn"] + data[base]["BENCHMARK"]["bertaqa_eu_global"] + data[base]["BENCHMARK"]["bertaqa_eu_local"] + data[base]["BENCHMARK"]["eus_exams_eu"] + data[base]["BENCHMARK"]["eus_reading"] + data[base]["BENCHMARK"]["eus_trivia"] + data[base]["BENCHMARK"]["mmlu_eu"] + data[base]["BENCHMARK"]["piqa_eu_mc"]) / 10
            latex = f"& {val1 * 100:.2f} \\raisebox{{-1.25ex}}{{\\hspace{{0.1em}}{{\\scriptsize {val2 * 100:.2f}}}}} & {val3 * 100:.2f} & {val4 * 100:.2f} \\raisebox{{-1.25ex}}{{\\hspace{{0.1em}}{{\\scriptsize {val5 * 100:.2f}}}}} & {val6 * 100:.2f} & {val7 * 100:.2f} & "
        else:
            latex = "& - & - & - & - & - & "
        assert instruct != ""
        if "gpt" in instruct:
            val1 = "-"
            val3 = "-"
            val5 = "-"
            val2 = np.mean(data[instruct]["BENCHMARK"]["ab-instruct_BL2MP"])
            val4 = np.mean(data[instruct]["BENCHMARK"]["ab-instruct_BAT"])
            latex = f"{latex}{val1} & {val2 * 100:.2f} & {val3} & {val4 * 100:.2f} & {val5} \\\\"
        else:
            if "THINKING" not in data[instruct]:
                val1 = np.mean(data[instruct]["LOGPROBS"]["bl2mp-inst"])
                val3 = np.mean(data[instruct]["LOGPROBS"]["bat-inst"])
                val5 = (data[instruct]["BENCHMARK"]["arc_eu_challenge_mc"] + data[instruct]["BENCHMARK"]["arc_eu_easy_mc"] + data[instruct]["BENCHMARK"]["belebele_eus_Latn"] + data[instruct]["BENCHMARK"]["bertaqa_eu_global"] + data[instruct]["BENCHMARK"]["bertaqa_eu_local"] + data[instruct]["BENCHMARK"]["eus_exams_eu"] + data[instruct]["BENCHMARK"]["eus_reading"] + data[instruct]["BENCHMARK"]["eus_trivia"] + data[instruct]["BENCHMARK"]["mmlu_eu"] + data[instruct]["BENCHMARK"]["piqa_eu_mc"]) / 10
                val2 = np.mean(data[instruct]["BENCHMARK"]["ab-instruct_BL2MP"])
                val4 = np.mean(data[instruct]["BENCHMARK"]["ab-instruct_BAT"])

                latex = f"{latex}{val1 * 100:.2f} & {val2 * 100:.2f} & {val3 * 100:.2f} & {val4 * 100:.2f} & {val5 * 100:.2f} \\\\"
            else:
                val1 = np.mean(data[instruct]["LOGPROBS"]["bl2mp-inst"])
                val2 = np.mean(data[instruct]["BENCHMARK"]["ab-instruct_BL2MP"])
                val3 = np.mean(data[instruct]["THINKING"]["ab-instruct_BL2MP-THINKING-SAMPLING"])

                val4 = np.mean(data[instruct]["LOGPROBS"]["bat-inst"])
                val5 = np.mean(data[instruct]["BENCHMARK"]["ab-instruct_BAT"])
                val6 = np.mean(data[instruct]["THINKING"]["ab-instruct_BAT-THINKING-SAMPLING"])

                val7 = (data[instruct]["BENCHMARK"]["arc_eu_challenge_mc"] + data[instruct]["BENCHMARK"]["arc_eu_easy_mc"] + data[instruct]["BENCHMARK"]["belebele_eus_Latn"] + data[instruct]["BENCHMARK"]["bertaqa_eu_global"] + data[instruct]["BENCHMARK"]["bertaqa_eu_local"] + data[instruct]["BENCHMARK"]["eus_exams_eu"] + data[instruct]["BENCHMARK"]["eus_reading"] + data[instruct]["BENCHMARK"]["eus_trivia"] + data[instruct]["BENCHMARK"]["mmlu_eu"] + data[instruct]["BENCHMARK"]["piqa_eu_mc"]) / 10

                latex = f"{latex}{val1 * 100:.2f} & {val2 * 100:.2f} \\raisebox{{-1.25ex}}{{\\hspace{{0.1em}}{{\\scriptsize {val3 * 100:.2f}}}}} & {val4 * 100:.2f} & {val5 * 100:.2f} \\raisebox{{-1.25ex}}{{\\hspace{{0.1em}}{{\\scriptsize {val6 * 100:.2f}}}}} & {val7 * 100:.2f} \\\\"

            
        print(instruct, latex)

del pairs, pair, base, instruct, latex, val1, val2, val3, val4, val5, val6, val7

In [ ]:
pairs = [
    [
        ["HiTZ/BOPV_60M-2-new", "HiTZ/berria_60M-2-new", "HiTZ/chimicron_60M-2-new", "HiTZ/ehkirola_60M-2-new", "HiTZ/instituzionalak_60M-2-new", "HiTZ/txioak_60M-2-new"],
        ["HiTZ/exp_2_101-BOPV_60M-2-new", "HiTZ/exp_2_101-berria_60M-2-new", "HiTZ/exp_2_101-chimicron_60M-2-new", "HiTZ/exp_2_101-ehkirola_60M-2-new", "HiTZ/exp_2_101-instituzionalak_60M-2-new", "HiTZ/exp_2_101-txioak_60M-2-new"]
    ],
]

for pair in pairs:
    for base, instruct in zip(*pair):
        if base != "":
            val1 = np.mean(data[base]["LOGPROBS"]["bl2mp-ctxt"])
            val2 = np.mean(data[base]["LOGPROBS"]["bl2mp-sent"])
            val3 = np.mean(data[base]["BENCHMARK"]["ab-base_BL2MP"])
            val4 = np.mean(data[base]["LOGPROBS"]["bat-ctxt"])
            val5 = np.mean(data[base]["LOGPROBS"]["bat-sent"])
            val6 = np.mean(data[base]["BENCHMARK"]["ab-base_BAT"])
            val7 = (data[base]["BENCHMARK"]["arc_eu_challenge_mc"] + data[base]["BENCHMARK"]["arc_eu_easy_mc"] + data[base]["BENCHMARK"]["belebele_eus_Latn"] + data[base]["BENCHMARK"]["bertaqa_eu_global"] + data[base]["BENCHMARK"]["bertaqa_eu_local"] + data[base]["BENCHMARK"]["eus_exams_eu"] + data[base]["BENCHMARK"]["eus_reading"] + data[base]["BENCHMARK"]["eus_trivia"] + data[base]["BENCHMARK"]["mmlu_eu"] + data[base]["BENCHMARK"]["piqa_eu_mc"]) / 10
            latex = f"& {val1 * 100:.2f} \\raisebox{{-1.25ex}}{{\\hspace{{0.1em}}{{\\scriptsize {val2 * 100:.2f}}}}} & {val3 * 100:.2f} & {val4 * 100:.2f} \\raisebox{{-1.25ex}}{{\\hspace{{0.1em}}{{\\scriptsize {val5 * 100:.2f}}}}} & {val6 * 100:.2f} & {val7 * 100:.2f} & "
        else:
            latex = "& - & - & - & - & - & "
        assert instruct != ""
        if "gpt" in instruct:
            val1 = "-"
            val3 = "-"
            val5 = "-"
            val2 = np.mean(data[instruct]["BENCHMARK"]["ab-instruct_BL2MP"])
            val4 = np.mean(data[instruct]["BENCHMARK"]["ab-instruct_BAT"])
            latex = f"{latex}{val1} & {val2 * 100:.2f} & {val3} & {val4 * 100:.2f} & {val5} \\\\"
        else:
            if "THINKING" not in data[instruct]:
                val1 = np.mean(data[instruct]["LOGPROBS"]["bl2mp-inst"])
                val3 = np.mean(data[instruct]["LOGPROBS"]["bat-inst"])
                val5 = (data[instruct]["BENCHMARK"]["arc_eu_challenge_mc"] + data[instruct]["BENCHMARK"]["arc_eu_easy_mc"] + data[instruct]["BENCHMARK"]["belebele_eus_Latn"] + data[instruct]["BENCHMARK"]["bertaqa_eu_global"] + data[instruct]["BENCHMARK"]["bertaqa_eu_local"] + data[instruct]["BENCHMARK"]["eus_exams_eu"] + data[instruct]["BENCHMARK"]["eus_reading"] + data[instruct]["BENCHMARK"]["eus_trivia"] + data[instruct]["BENCHMARK"]["mmlu_eu"] + data[instruct]["BENCHMARK"]["piqa_eu_mc"]) / 10
                val2 = np.mean(data[instruct]["BENCHMARK"]["ab-instruct_BL2MP"])
                val4 = np.mean(data[instruct]["BENCHMARK"]["ab-instruct_BAT"])

                latex = f"{latex}{val1 * 100:.2f} & {val2 * 100:.2f} & {val3 * 100:.2f} & {val4 * 100:.2f} & {val5 * 100:.2f} \\\\"
            else:
                val1 = np.mean(data[instruct]["LOGPROBS"]["bl2mp-inst"])
                val2 = np.mean(data[instruct]["BENCHMARK"]["ab-instruct_BL2MP"])
                val3 = np.mean(data[instruct]["THINKING"]["ab-instruct_BL2MP-THINKING-SAMPLING"])

                val4 = np.mean(data[instruct]["LOGPROBS"]["bat-inst"])
                val5 = np.mean(data[instruct]["BENCHMARK"]["ab-instruct_BAT"])
                val6 = np.mean(data[instruct]["THINKING"]["ab-instruct_BAT-THINKING-SAMPLING"])

                val7 = (data[instruct]["BENCHMARK"]["arc_eu_challenge_mc"] + data[instruct]["BENCHMARK"]["arc_eu_easy_mc"] + data[instruct]["BENCHMARK"]["belebele_eus_Latn"] + data[instruct]["BENCHMARK"]["bertaqa_eu_global"] + data[instruct]["BENCHMARK"]["bertaqa_eu_local"] + data[instruct]["BENCHMARK"]["eus_exams_eu"] + data[instruct]["BENCHMARK"]["eus_reading"] + data[instruct]["BENCHMARK"]["eus_trivia"] + data[instruct]["BENCHMARK"]["mmlu_eu"] + data[instruct]["BENCHMARK"]["piqa_eu_mc"]) / 10

                latex = f"{val1 * 100:.2f} & {val2 * 100:.2f} \\raisebox{{-1.25ex}}{{\\hspace{{0.1em}}{{\\scriptsize {val3 * 100:.2f}}}}} & {val4 * 100:.2f} & {val5 * 100:.2f} \\raisebox{{-1.25ex}}{{\\hspace{{0.1em}}{{\\scriptsize {val6 * 100:.2f}}}}} & {val7 * 100:.2f} \\\\"

            
        print(base, latex)

del pairs, pair, base, instruct, latex, val1, val2, val3, val4, val5, val6, val7

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import textwrap
from matplotlib.lines import Line2D

# ==========================================
# CONFIGURATION & DATA (Berria & EHKirola + New Origin Set)
# ==========================================
MASTER_DATA = {
    'A': {
        'xticks': np.array([2, 5, 8]),
        'xticklabels': ['Direct Probability Measurements', 'Instruction Probability Measurements', 'A/B Multiple-Choice'],
        'ylim': (62, 78),
        'xlim': (0, 10),
        'dividers': [3.5, 6.5],
        'lines': [
            # --- Circle Origin Set ---
            (np.array([(np.mean(data["HiTZ/Latxa-Llama-3.1-8B"]["LOGPROBS"]["bat-sent"])  * 100, np.mean(data["HiTZ/berria_60M-2-new"]["LOGPROBS"]["bat-sent"]) * 100), (np.mean(data["HiTZ/Latxa-Llama-3.1-8B"]["LOGPROBS"]["bat-ctxt"]) * 100, np.mean(data["HiTZ/berria_60M-2-new"]["LOGPROBS"]["bat-ctxt"]) * 100), (np.mean(data["HiTZ/Latxa-Llama-3.1-8B"]["BENCHMARK"]["ab-base_BAT"]) * 100, np.mean(data["HiTZ/berria_60M-2-new"]["BENCHMARK"]["ab-base_BAT"]) * 100)]), "Berria (Circle)", 'o'),
            (np.array([(np.mean(data["HiTZ/Latxa-Llama-3.1-8B"]["LOGPROBS"]["bat-sent"])  * 100, np.mean(data["HiTZ/ehkirola_60M-2-new"]["LOGPROBS"]["bat-sent"]) * 100), (np.mean(data["HiTZ/Latxa-Llama-3.1-8B"]["LOGPROBS"]["bat-ctxt"]) * 100, np.mean(data["HiTZ/ehkirola_60M-2-new"]["LOGPROBS"]["bat-ctxt"]) * 100), (np.mean(data["HiTZ/Latxa-Llama-3.1-8B"]["BENCHMARK"]["ab-base_BAT"]) * 100, np.mean(data["HiTZ/ehkirola_60M-2-new"]["BENCHMARK"]["ab-base_BAT"]) * 100)]), "EHKirola (Circle)", 'o'),
            # --- Triangle Origin Set ---
            (np.array([(0.00, 0.00), (np.mean(data["HiTZ/Latxa-Llama-3.1-8B-Instruct"]["LOGPROBS"]["bat-inst"]) * 100, np.mean(data["HiTZ/exp_2_101-berria_60M-2-new"]["LOGPROBS"]["bat-inst"]) * 100), (np.mean(data["HiTZ/Latxa-Llama-3.1-8B-Instruct"]["BENCHMARK"]["ab-instruct_BAT"]) * 100, np.mean(data["HiTZ/exp_2_101-berria_60M-2-new"]["BENCHMARK"]["ab-instruct_BAT"]) * 100)]), "Berria (Triangle)", '^'),
            (np.array([(0.00, 0.00), (np.mean(data["HiTZ/Latxa-Llama-3.1-8B-Instruct"]["LOGPROBS"]["bat-inst"]) * 100, np.mean(data["HiTZ/exp_2_101-ehkirola_60M-2-new"]["LOGPROBS"]["bat-inst"]) * 100), (np.mean(data["HiTZ/Latxa-Llama-3.1-8B-Instruct"]["BENCHMARK"]["ab-instruct_BAT"]) * 100, np.mean(data["HiTZ/exp_2_101-ehkirola_60M-2-new"]["BENCHMARK"]["ab-instruct_BAT"]) * 100)]), "EHKirola (Triangle)", '^'),
        ],
    },
}

plt.rcParams.update({
    'font.family': 'sans-serif',
    'axes.edgecolor': '#E0E0E0',
    'axes.linewidth': 1.5,
    'axes.labelcolor': '#333333',
    'xtick.color': '#555555',
    'ytick.color': '#555555',
    'text.color': '#1A1A1A',
})

# Color palette updated: Berria is Green, EHKirola is Red
COLORS = ['#2E7D32', '#D32F2F', '#2E7D32', '#D32F2F']

# ==========================================
# PLOTTING
# ==========================================
fig, ax = plt.subplots(1, 1, figsize=(10, 7.5), facecolor='#F8F9FA') 
subplot_mapping = {'A': ax}

for key, cfg in MASTER_DATA.items():
    current_ax = subplot_mapping[key]
    current_ax.set_facecolor('#FFFFFF') 
    
    # Outer boundaries and dividers
    current_ax.axvline(cfg['xlim'][0], color='#A0A0A0', linewidth=2.5, zorder=1)
    current_ax.axvline(cfg['xlim'][1], color='#A0A0A0', linewidth=2.5, zorder=1)
    for div in cfg['dividers']:
        current_ax.axvline(div, color='#CCCCCC', linestyle=':', linewidth=1.5, zorder=1)

    # Plot lines dynamically based on provided configuration entries
    for i, (y_vals, label, marker) in enumerate(cfg['lines']):
        plot_kwargs = {
            'color': COLORS[i], 'marker': marker, 'linewidth': 2.5,
            'markersize': 8, 'markerfacecolor': COLORS[i],
            'markeredgecolor': '#FFFFFF', 'markeredgewidth': 1.5, 
            'zorder': 4, 'alpha': 0.9, 'markevery': [-1]
        }

        for j in range(3):
            if marker == '^' and j == 0:
                continue
            x_start, x_end = cfg['xticks'][j] - 1, cfg['xticks'][j] + 1
            current_ax.plot([x_start, x_end], y_vals[j], **plot_kwargs)

    # Highlight common starting baselines
    origin_indices = [0, 2]
    origin_markers = ['o', '^']
    
    for idx, marker_shape in zip(origin_indices, origin_markers):
        base_y_vals = cfg['lines'][idx][0] 
        for j in range(3):
            if marker_shape == '^' and j == 0:
                continue
            start_x, start_y = cfg['xticks'][j] - 1, base_y_vals[j][0]
            current_ax.plot([start_x, start_x + 2], [start_y, start_y], color='#888888', linestyle=':', linewidth=1.5, zorder=2, alpha=0.7)
            current_ax.plot(start_x, start_y, marker=marker_shape, markerfacecolor='#FFFFFF', 
                    markeredgecolor='#333333', markeredgewidth=3, markersize=11, zorder=6)

    # Styling axes
    current_ax.grid(True, axis='y', linestyle='--', alpha=0.4, color='#A0A0A0', linewidth=0.7, zorder=0)
    for spine in ['top', 'right']: 
        current_ax.spines[spine].set_visible(False)
        
    current_ax.set_ylim(cfg['ylim'])
    current_ax.set_xlim(cfg['xlim'])
    
    # Apply text wrapping to the x-axis labels with INCREASED FONT
    current_ax.set_xticks(cfg['xticks'])
    wrapped_labels = [textwrap.fill(lbl, width=15) for lbl in cfg['xticklabels']]
    current_ax.set_xticklabels(wrapped_labels, fontsize=20, fontweight='bold')
    
    current_ax.set_yticks(np.arange(cfg['ylim'][0], cfg['ylim'][1] + 0.1, 2))
    
    # INCREASE Y-AXIS TICK SIZE
    current_ax.tick_params(axis='y', labelsize=18)

# Labels - INCREASED FONT
ax.set_ylabel('BAT Accuracy Score', labelpad=14, fontsize=25, fontweight='bold')

# Unified legend cleanly splitting Brand/Color and Starting Points/Origins
legend_elements = [
    # Colors
    Line2D([0], [0], color='#2E7D32', lw=4, label='Berria'),
    Line2D([0], [0], color='#D32F2F', lw=4, label='EHKirola'),
    # Origins matching the styled canvas markers
    Line2D([0], [0], color='none', marker='o', markersize=14, 
           markerfacecolor='#FFFFFF', markeredgecolor='#333333', markeredgewidth=3, label='Latxa-Llama-3.1-8B'),
    Line2D([0], [0], color='none', marker='^', markersize=14, 
           markerfacecolor='#FFFFFF', markeredgecolor='#333333', markeredgewidth=3, label='Latxa-Llama-3.1-8B-Instruct'),
]

# ==========================================
# LAYOUT & SPACING MANAGEMENT
# ==========================================
# Adjusted from fig.legend to ax.legend to make coordinates relative to the graph
ax.legend(
    handles=legend_elements, 
    loc='lower center', 
    bbox_to_anchor=(0.5, 1.02),  # Anchors just above the axes (1.0 is the exact top border)
    ncol=2, 
    frameon=True, 
    facecolor='#FFFFFF', 
    edgecolor='#E0E0E0',
    fontsize=22,                 # INCREASED FONT SIZE (was 14)
    labelspacing=0.8,
    columnspacing=1.5
)

# You can now give a bit more room to the top since the gap is removed
plt.subplots_adjust(top=0.90, bottom=0.10, left=0.12, right=0.95)

plt.savefig('curriculum.pdf', format="pdf", bbox_inches='tight')
print("Successfully generated plot with the legend on top.")